In [1]:
#!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
# lets create a smaple document 
sample_doc = Document(
    page_content="Hello World",
    metadata={"source" : "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Sample for Text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/Python.txt", encoding = "utf-8")

C:\Users\Daman Jeet Kaur\AppData\Local\Temp\ipykernel_21756\1830763608.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [ ]:
# Sample for PDF data 
from langchain_community.document_loaders.pdf import PyMuPDFLoader
loader = PyMuPDFLoader("data/research.pdf")
document = loader.load()
document

## Ingestion Pipeline

In [10]:
# first, convert Data => Documents
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [11]:
# a func to load all PDFs & convert into documents 
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path) :
        if filename.lower().endswith(".pdf") :
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1
            
    print("total pdfs : ", num_docs)
    print("total_pages : ", len(all_docs))
    return all_docs

In [12]:
all_pdf_docs = load_all_pdfs()

total pdfs :  2
total_pages :  32


In [13]:
type(all_pdf_docs[1])

langchain_core.documents.base.Document

### Chunks

In [14]:
# chunks 
#!pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
# a func to split docs into chunks
def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [16]:
chunks = split_docs(all_pdf_docs)

In [17]:
len(chunks)

321

### Embedding

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
# a func to create embeddings
class EmbeddingManager :
    def __init__(self, model_name = "all-MiniLM-L6-v2") :
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    
    def generate_embeddings(self, text) :
        embeddings = self.model.encode(text)
        print("embeddings shape : ", embeddings.shape)
        return embeddings

In [20]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions =  384


C:\Users\Daman Jeet Kaur\AppData\Local\Temp\ipykernel_21756\2705356068.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())


### Vector Store

In [21]:
import chromadb 
import uuid

In [22]:
class VectorStoreManager :
    def __init__(self, persist_directory = "data/vector_store", collection_name = "pdf_documents") :
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None 

        self._initialize_store()

    def _initialize_store(self) :
        os.makedirs(self.persist_directory, exist_ok = True)
        
        # create a client 
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection : ", self.collection_name)
        print("docs in colleciton : ", self.collection.count())

    def add_documents(self, documents, embeddings)  :
        if len(documents) != len(embeddings) :
            raise ValueError("num of documents does not match num of embeddings")

        # we've to store -> ids, embeding, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)) :
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            # not we insert these 4 values as a single unit in our collection 
            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total documents added in vector store : ", len(documents_content))
        print("docs in collection : ", self.collection.count())

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection :  pdf_documents
docs in colleciton :  0


In [24]:
# chunks => embeddings => add to the vector store
texts = [doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

embeddings shape :  (321, 384)
total documents added in vector store :  321
docs in collection :  321


## Retrieval Pipeline

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriever :
    def __init__(self, embedding_manager, vector_store) :
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0) :
        # query => embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search 
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0] :
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0] # these are the cosine distances

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)) :
                similarity_score = 1 - distance

                if similarity_score >= score_threshold :
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "distance" : distance,
                        "similarity_socre" : similarity_score,
                        "rank" : i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")  
                    
        else :
            print("no documents found")

        return retrieved_docs    

In [27]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [28]:
rag_retriever.retrieve("What is encoder decoder")

embeddings shape :  (1, 384)
retrieved 4 documents


[{'id': 'doc_86e8614d-a7ad-4946-b1de-c5c7a4eb7d3c',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\ndhillondaman2003@gmail.com',
  'metadata': {'eventtype': 'Poster',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'type': 'Conference Proceedings',
   'page': 6,
   'publisher': 'Curran Associates, Inc.',
   'creator': 'PyPDF',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensi

# Integrate with LLMs

## OpenAI-GPT

In [29]:
#!pip install langchain-openai

In [30]:
import getpass
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = getpass.getpass("Router key (sk-air-v1-...): ").strip()

llm = ChatOpenAI(
    openai_api_key = OPENAI_API_KEY,
    model = "openai/gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024,
    openai_api_base = "https://api.airouter.in/v1"
)

Router key (sk-air-v1-...):  ········


In [31]:
# a func to generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k = 3) :
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""
    if not context :
        print("we found no relevent context for the query")

    # context + query = prompt 
    prompt = f""" use the given context to generate an answer for the query
                Context : {context}
                Query : {query} """
    response = llm.invoke(prompt)
    return response

In [32]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

embeddings shape :  (1, 384)
retrieved 3 documents


In [33]:
print(answer.content)

The encoder-decoder is an architecture with two main parts:

- Encoder: a stack of 6 identical layers.
  - Each layer has:
    - a multi-head self-attention sub-layer
    - a position-wise feed-forward sub-layer
  - Residual connections and layer normalization are applied around each sub-layer.
  - The outputs have dimension d_model = 512.

- Decoder: also a stack of 6 identical layers.
  - Each layer has:
    - a masked multi-head self-attention sub-layer
    - an encoder-decoder attention sub-layer that attends over the encoder’s output
    - a position-wise feed-forward sub-layer
  - It also uses residual connections and layer normalization.

In short, the encoder processes the input sequence into representations, and the decoder uses those representations to generate the output sequence.


## Groq

In [34]:
# !pip install langchain-groq

In [35]:
import getpass
from langchain_groq import ChatGroq

GROQ_API_KEY = getpass.getpass("Groq key (gsk_...): ").strip()

llm = ChatGroq(
    groq_api_key = GROQ_API_KEY, 
    model = "qwen/qwen3.8-27b", 
    temperature = 0.1, 
    max_tokens = 1024
)

Groq key (gsk_...):  ········


In [36]:
# generate our retreival-augmented output 
def generate_output(query, retriever, llm, top_k = 3) :
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context :
        print("we found no relevent context for the given query")

    # context + query => prompt 
    prompt = f""" use the given context to generate the answer for the query
                Context : {context}
                Query : {query} """

    response = llm.invoke([prompt.format(context = context, query = query)]) # expects a list as a prompt
    return response.content

In [37]:
answer = generate_output("what is RAG?", rag_retriever, llm)

embeddings shape :  (1, 384)
retrieved 3 documents


In [38]:
print(answer)

Based on the provided context, **RAG** (Retrieval-Augmented Generation) is a field that has experienced swift growth and is primarily focused on its **integration within Large Language Models (LLMs)**.

The context describes RAG through the following key aspects:

*   **Evolutionary Paradigms:** It has evolved through different stages, including **naive RAG** and **effective RAG frameworks**.
*   **Scope of Study:** The survey maps out the RAG process, charting its evolution and anticipated future paths.
*   **Evaluation:** It involves specific assessment methods, covering 26 tasks, nearly 50 datasets, and various evaluation objectives, metrics, benchmarks, and tools.
*   **Goal:** The field aims to tackle current challenges and anticipate future directions, with a focus on technical paradigms and research methods.

In essence, RAG is a rapidly developing area of AI research centered on enhancing LLMs through retrieval-augmented techniques, with a structured evolution from naive to mor